# Loss comparison on real lineages — CL / RINCE / LRCL / Hybrid

Trace the positive/negative structure of each contrastive loss on the SAME real-lineage batch, to
make the differences concrete. Four losses:

- **CL (InfoNCE)** — plain MERU/CLIP: 1 positive (own species text), every other species a negative
  (incl. same-genus → *false negatives*).
- **RINCE** (Hoffmann 2022) — graded by shared taxonomic depth; per tier d, positives = depth==d
  (exactly-d), over *instances*, multi-positive SupCon. No dedup, asymmetric.
- **LRCL** (Tao 2026) — per rank, image↔*deduplicated* class-text prototype, symmetric I→T + T→I.
  Same-species mate counted at every rank.
- **Hybrid** (ours) — RINCE's exactly-d grading + LRCL's deduped-prototype symmetric I↔T.

Dedup = unique text set (both directions, text axis). Grading (exactly-d) = T→I only (the only
direction with a multi-positive image set 𝒫ₖ to refine).

In [1]:
# real-lineage batch: img0 = query (Lepidoptera/Gelechiidae/Dichomeris/santarosensis)
lin = [
 ("Lepidoptera","Gelechiidae","Dichomeris","Dichomeris santarosensis"),  # img0  query
 ("Lepidoptera","Gelechiidae","Dichomeris","Dichomeris santarosensis"),  # img1  SAME species
 ("Lepidoptera","Gelechiidae","Dichomeris","Dichomeris malaisei"),       # img2  same GENUS
 ("Lepidoptera","Gelechiidae","Sinoe","Sinoe bioLep1"),                  # img3  same FAMILY
 ("Lepidoptera","Cosmopterigidae","Cosmo","Cosmo x"),                    # img4  same ORDER
 ("Diptera","Culicidae","Aedes","Aedes aegypti"),                        # img5  UNRELATED
]
ranks = ["order","family","genus","species"]
rel = {0:"query",1:"SAME species",2:"same GENUS",3:"same FAMILY",4:"same ORDER",5:"UNRELATED"}
def shared_depth(a,b):
    d=0
    for x,y in zip(a,b):
        if x==y: d+=1
        else: break
    return d
depth = [shared_depth(lin[0],lin[j]) for j in range(6)]
print("BATCH (query = img0):")
for k,l in enumerate(lin):
    print(f"  img{k}: {l[0][:4]}/{l[1][:6]}/{l[2][:8]}/{l[3].split()[-1][:10]:10}  [{rel[k]}]  shared-depth w/ img0 = {depth[k]}")

# CUMULATIVE labels (what the ACTUAL loss dedups on — prefix lineage, not bare value).
# Two same-named genera in different families get DIFFERENT cumulative strings -> not wrongly merged.
def cumul(j, di):  # cumulative label of img j through rank index di (0=order..3=species)
    return " ".join(lin[j][:di+1])
print("\ncumulative (prefix) labels — the real dedup keys:")
for di,r in enumerate(ranks):
    print(f"  {r}: " + " | ".join(sorted(set(cumul(j,di) for j in range(6)))))

BATCH (query = img0):
  img0: Lepi/Gelech/Dichomer/santarosen  [query]  shared-depth w/ img0 = 4
  img1: Lepi/Gelech/Dichomer/santarosen  [SAME species]  shared-depth w/ img0 = 4
  img2: Lepi/Gelech/Dichomer/malaisei    [same GENUS]  shared-depth w/ img0 = 3
  img3: Lepi/Gelech/Sinoe/bioLep1     [same FAMILY]  shared-depth w/ img0 = 2
  img4: Lepi/Cosmop/Cosmo/x           [same ORDER]  shared-depth w/ img0 = 1
  img5: Dipt/Culici/Aedes/aegypti     [UNRELATED]  shared-depth w/ img0 = 0

cumulative (prefix) labels — the real dedup keys:
  order: Diptera | Lepidoptera
  family: Diptera Culicidae | Lepidoptera Cosmopterigidae | Lepidoptera Gelechiidae
  genus: Diptera Culicidae Aedes | Lepidoptera Cosmopterigidae Cosmo | Lepidoptera Gelechiidae Dichomeris | Lepidoptera Gelechiidae Sinoe
  species: Diptera Culicidae Aedes Aedes aegypti | Lepidoptera Cosmopterigidae Cosmo Cosmo x | Lepidoptera Gelechiidae Dichomeris Dichomeris malaisei | Lepidoptera Gelechiidae Dichomeris Dichomeris santaros

## CL (InfoNCE) — 1 positive, all other species are negatives (false negatives!)

In [2]:
print("query = img0. Positive = its OWN species text. Negatives = ALL other texts.")
print(f"  POSITIVE:  img0's species text (Dichomeris santarosensis)")
print(f"  NEGATIVES: every OTHER sample's species text:")
for j in range(1,6):
    fn = "  <-- FALSE NEGATIVE (same species!)" if depth[j]==4 else ""
    print(f"     img{j} species text [{rel[j]}]{fn}")
print("\n  => img1 (same species) is treated as a NEGATIVE: the false-negative problem.")

query = img0. Positive = its OWN species text. Negatives = ALL other texts.
  POSITIVE:  img0's species text (Dichomeris santarosensis)
  NEGATIVES: every OTHER sample's species text:
     img1 species text [SAME species]  <-- FALSE NEGATIVE (same species!)
     img2 species text [same GENUS]
     img3 species text [same FAMILY]
     img4 species text [same ORDER]
     img5 species text [UNRELATED]

  => img1 (same species) is treated as a NEGATIVE: the false-negative problem.


## RINCE — per tier d (positives = depth EXACTLY d), over instances, multi-positive

In [3]:
rank_at = {4:"species",3:"genus",2:"family",1:"order"}
for d in [4,3,2,1]:
    pos  = [j for j in range(6) if depth[j]==d]
    excl = [j for j in range(6) if depth[j]>d]
    neg  = [j for j in range(6) if 0<=depth[j]<d]
    print(f"tier d={d} ({rank_at[d]}):  tau={'hot' if d<4 else 'cold'}")
    print(f"   POSITIVES (depth=={d}): {['img'+str(j) for j in pos]}  (multi-positive, summed-in-log)")
    print(f"   negatives (depth<{d}):  {['img'+str(j) for j in neg]}")
    print(f"   EXCLUDED (depth>{d}, masked -inf): {['img'+str(j) for j in excl]}")
print("\n  => at the GENUS tier, same-species img0/img1 are EXCLUDED; positive is only img2.")
print("     contrasts INSTANCES (no dedup), one direction (I->bank).")

tier d=4 (species):  tau=cold
   POSITIVES (depth==4): ['img0', 'img1']  (multi-positive, summed-in-log)
   negatives (depth<4):  ['img2', 'img3', 'img4', 'img5']
   EXCLUDED (depth>4, masked -inf): []
tier d=3 (genus):  tau=hot
   POSITIVES (depth==3): ['img2']  (multi-positive, summed-in-log)
   negatives (depth<3):  ['img3', 'img4', 'img5']
   EXCLUDED (depth>3, masked -inf): ['img0', 'img1']
tier d=2 (family):  tau=hot
   POSITIVES (depth==2): ['img3']  (multi-positive, summed-in-log)
   negatives (depth<2):  ['img4', 'img5']
   EXCLUDED (depth>2, masked -inf): ['img0', 'img1', 'img2']
tier d=1 (order):  tau=hot
   POSITIVES (depth==1): ['img4']  (multi-positive, summed-in-log)
   negatives (depth<1):  ['img5']
   EXCLUDED (depth>1, masked -inf): ['img0', 'img1', 'img2', 'img3']

  => at the GENUS tier, same-species img0/img1 are EXCLUDED; positive is only img2.
     contrasts INSTANCES (no dedup), one direction (I->bank).


## LRCL — per rank, image ↔ DEDUPED class-text prototype, symmetric I→T + T→I

In [4]:
for ri,r in enumerate(ranks):
    labels=[cumul(j,ri) for j in range(6)]          # CUMULATIVE (the real dedup key)
    uniq=list(dict.fromkeys(labels))
    same=[j for j in range(6) if labels[j]==labels[0]]
    short=lambda s: s.split()[-1]                    # show last token for readability
    print(f"rank {r}: UNIQUE cumulative prototypes 𝒦 = {[short(u) for u in uniq]}")
    print(f"   img0's {r} = {short(labels[0])!r}; image group 𝒫 (T->I positives) = {['img'+str(j) for j in same]}")
    print(f"   I->T: img0 -> its {r} cumulative prototype; negatives = other {len(uniq)-1} unique prototypes")
    print(f"   T->I: prototype -> ALL images in group (incl. same-species img1)")
print("\n  => DEDUP on CUMULATIVE text (both directions). img1 counted at EVERY rank (no exactly-d).")

rank order: UNIQUE cumulative prototypes 𝒦 = ['Lepidoptera', 'Diptera']
   img0's order = 'Lepidoptera'; image group 𝒫 (T->I positives) = ['img0', 'img1', 'img2', 'img3', 'img4']
   I->T: img0 -> its order cumulative prototype; negatives = other 1 unique prototypes
   T->I: prototype -> ALL images in group (incl. same-species img1)
rank family: UNIQUE cumulative prototypes 𝒦 = ['Gelechiidae', 'Cosmopterigidae', 'Culicidae']
   img0's family = 'Gelechiidae'; image group 𝒫 (T->I positives) = ['img0', 'img1', 'img2', 'img3']
   I->T: img0 -> its family cumulative prototype; negatives = other 2 unique prototypes
   T->I: prototype -> ALL images in group (incl. same-species img1)
rank genus: UNIQUE cumulative prototypes 𝒦 = ['Dichomeris', 'Sinoe', 'Cosmo', 'Aedes']
   img0's genus = 'Dichomeris'; image group 𝒫 (T->I positives) = ['img0', 'img1', 'img2']
   I->T: img0 -> its genus cumulative prototype; negatives = other 3 unique prototypes
   T->I: prototype -> ALL images in group (incl. sam

## Hybrid — RINCE exactly-d grading + LRCL dedup + symmetry

Dedup (text, both dirs) like LRCL; per-tier temperature + exactly-d exclusion in T→I (drop the
representative's finer-rank mates from the prototype's image group).

In [5]:
for di,r in enumerate(ranks):
    d=di+1; finer = ranks[di+1] if di+1<len(ranks) else None
    labels=[cumul(j,di) for j in range(6)]           # CUMULATIVE dedup key
    uniq=list(dict.fromkeys(labels))
    short=lambda s: s.split()[-1]
    print(f"tier d={d} ({r}): DEDUPED cumulative prototypes 𝒦 = {[short(u) for u in uniq]}   (tau {'hotter' if d<4 else 'coldest'})")
    print(f"   I->T (LRCL): img0 -> its {r} cumulative prototype; negatives = other unique prototypes")
    grp=[j for j in range(6) if labels[j]==labels[0]]
    if finer is not None:
        rep_finer = cumul(grp[0], di+1)              # representative's finer CUMULATIVE label
        kept=[j for j in grp if cumul(j,di+1)!=rep_finer]
        dropped=[j for j in grp if cumul(j,di+1)==rep_finer]
        print(f"   T->I (exactly-d): group = {['img'+str(j) for j in grp]}; DROP rep's {finer}-mates {['img'+str(j) for j in dropped]}")
        print(f"        -> kept positives: {['img'+str(j) for j in kept] or '(none -> fall back to full group)'}")
    else:
        print(f"   T->I: species group = {['img'+str(j) for j in grp]} (deepest, no exclusion)")
print("\n  => DEDUP on CUMULATIVE text (both dirs) + exactly-d (T->I). At genus: same-species img1 DROPPED.")

tier d=1 (order): DEDUPED cumulative prototypes 𝒦 = ['Lepidoptera', 'Diptera']   (tau hotter)
   I->T (LRCL): img0 -> its order cumulative prototype; negatives = other unique prototypes
   T->I (exactly-d): group = ['img0', 'img1', 'img2', 'img3', 'img4']; DROP rep's family-mates ['img0', 'img1', 'img2', 'img3']
        -> kept positives: ['img4']
tier d=2 (family): DEDUPED cumulative prototypes 𝒦 = ['Gelechiidae', 'Cosmopterigidae', 'Culicidae']   (tau hotter)
   I->T (LRCL): img0 -> its family cumulative prototype; negatives = other unique prototypes
   T->I (exactly-d): group = ['img0', 'img1', 'img2', 'img3']; DROP rep's genus-mates ['img0', 'img1', 'img2']
        -> kept positives: ['img3']
tier d=3 (genus): DEDUPED cumulative prototypes 𝒦 = ['Dichomeris', 'Sinoe', 'Cosmo', 'Aedes']   (tau hotter)
   I->T (LRCL): img0 -> its genus cumulative prototype; negatives = other unique prototypes
   T->I (exactly-d): group = ['img0', 'img1', 'img2']; DROP rep's species-mates ['img0', 'img

## Summary table

| | positives | dedup | symmetric | exactly-d grading | false-neg fix |
|---|---|---|---|---|---|
| **CL** | 1 (own species) | no | symmetric I/T | no | ❌ (same-species = negative) |
| **RINCE** | multi (depth==d), instances | no | I→bank only | ✓ (excl depth>d) | ✓ via grading |
| **LRCL** | I→T 1 proto; T→I image group | text (both dirs) | ✓ | no | ✓ via dedup |
| **Hybrid** | deduped protos, symmetric | text (both dirs) | ✓ | ✓ (T→I only) | ✓ via dedup + grading |